# Lab 1 — Data Split

This notebook is the **single source of truth** for data preparation. It:

1. Loads the **1 K**, **25 K**, and **Video Games** review datasets.
2. Applies a reproducible **stratified train / validation / test split** (81 % / 9 % / 10 %).
3. Saves each split to CSV so that every downstream experiment notebook can load identical data without repeating any of this logic.

**Output files written to `../data/splits/`**

```
splits/
├── 1k_{train,val,test}.csv      ← binary sentiment  (Class: 0/1)
├── 25k_{train,val,test}.csv     ← binary sentiment  (Class: 0/1)
└── vg_{train,val,test}.csv      ← star rating       (Class: 1–5)
```

Each CSV has two columns: `Sentence` (raw text) and `Class`.

> **Run this notebook once before running any experiment notebook.**  
> Text is saved **raw** — model-specific preprocessing (e.g. stopword removal for TF-IDF) is applied in each experiment notebook.

## Imports & Configuration

In [1]:
import json
import pathlib

import numpy as np
import pandas as pd

from utils import stratified_split

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 1
np.random.seed(SEED)

# ── Split ratios ────────────────────────────────────────────────────────────
# Hold-out test: 10 %  →  remaining 90 % split 90/10 → train/val
# Effective split:  train ≈ 81 %  |  val ≈ 9 %  |  test = 10 %
TEST_SIZE = 0.10
VAL_SIZE  = 0.10   # fraction of the train+val pool

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR   = pathlib.Path('../data')
SPLITS_DIR = DATA_DIR / 'splits'
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

RAW_1K  = DATA_DIR / 'amazon_cells_labelled.txt'
RAW_25K = DATA_DIR / 'amazon_cells_labelled_LARGE_25K.txt'
RAW_VG  = DATA_DIR / 'Video_Games.json'

print(f'Splits will be saved to: {SPLITS_DIR.resolve()}')

Splits will be saved to: /root/D7047E/Lab1/data/splits


## Helper — Split & Save

Encapsulates the full pipeline for a single dataset: load → split → save.

In [2]:
def load_and_split(
    raw_path: pathlib.Path,
    prefix: str,
    test_size: float = TEST_SIZE,
    val_size: float = VAL_SIZE,
    seed: int = SEED,
) -> dict[str, pd.DataFrame]:
    """Load, split, and save one dataset.

    Parameters
    ----------
    raw_path : path to the raw tab-separated .txt file
    prefix   : short label used in output filenames, e.g. '1k' or '25k'
    test_size: fraction of the full dataset held out as test set
    val_size : fraction of the train+val pool used for validation
    seed     : random state for reproducible splits

    Returns
    -------
    dict with keys 'train', 'val', 'test', each a DataFrame.
    """
    # ── 1. Load ───────────────────────────────────────────────────────────────
    raw = pd.read_csv(
        raw_path,
        delimiter='\t',
        header=None,
        names=['Sentence', 'Class'],
    )
    print(f'[{prefix}] Loaded {len(raw):,} raw reviews')
    print(raw['Class'].value_counts().rename({0: 'negative', 1: 'positive'}).to_string())

    # ── 2. Split — stratified to preserve class balance ───────────────────────
    splits = stratified_split(
        data=raw,
        label_col='Class',
        test_size=test_size,
        val_size=val_size,
        seed=seed
    )

    # ── 3. Save ───────────────────────────────────────────────────────────────
    for name, df in splits.items():
        out = SPLITS_DIR / f'{prefix}_{name}.csv'
        df.to_csv(out, index=False)
        print(f'  Saved → {out}')

    return splits

## 1 K Dataset

In [3]:
splits_1k = load_and_split(RAW_1K, prefix='1k')

[1k] Loaded 1,000 raw reviews
Class
negative    500
positive    500
Stratified split — total: 1,000  |  seed=1
  train:        810  (81.0 %)  [0=405  1=405]
  val  :         90  ( 9.0 %)  [0=45  1=45]
  test :        100  (10.0 %)  [0=50  1=50]
  Saved → ../data/splits/1k_train.csv
  Saved → ../data/splits/1k_val.csv
  Saved → ../data/splits/1k_test.csv


In [4]:
splits_1k['train'].head()

,Sentence,Class
0,"Uncomfortable In the Ear, Don't use with LG VX...",0
1,I have two more years left in this contract an...,0
2,I'm still infatuated with this phone.,1
3,I exchanged the sony ericson z500a for this an...,1
4,I didn't think that the instructions provided ...,0


## 25 K Dataset

In [5]:
splits_25k = load_and_split(RAW_25K, prefix='25k')

[25k] Loaded 25,000 raw reviews
Class
positive    15116
negative     9884
Stratified split — total: 25,000  |  seed=1
  train:     20,250  (81.0 %)  [0=8,006  1=12,244]
  val  :      2,250  ( 9.0 %)  [0=890  1=1,360]
  test :      2,500  (10.0 %)  [0=988  1=1,512]
  Saved → ..\data\splits\25k_train.csv
  Saved → ..\data\splits\25k_val.csv
  Saved → ..\data\splits\25k_test.csv


In [6]:
splits_25k['train'].head()

,Sentence,Class
0,Do not place any orders that involves this sel...,0
1,"Without Blue oysters music,the sound track is ...",0
2,"After I set it up, I took a test shot and the ...",1
3,wheres the sex?? And can Plamer stop using the...,0
4,This book provides a nice review of Office 200...,1


## Video Games Dataset

Amazon product reviews for video games in JSONL format (one JSON object per line).
We extract `reviewText` → `Sentence` and `overall` (1–5 star rating) → `Class`,
dropping the ~1 700 records that have no review text.

In [7]:
# -- 1. Load --------------------------------------------------------------------
records = []
with open(RAW_VG, encoding="utf-8") as f:
    for line in f:
        rec  = json.loads(line)
        text = rec.get("reviewText")
        if isinstance(text, str):
            text = text.strip()
        if text:
            records.append({"Sentence": text, "Class": int(rec["overall"])})

vg_raw = pd.DataFrame(records)
del records  # free the intermediate list

# Drop any rows where Sentence is missing or blank (handles CSV round-trip NaN)
before = len(vg_raw)
vg_raw = vg_raw.dropna(subset=["Sentence"])
vg_raw = vg_raw[vg_raw["Sentence"].str.strip() != ""]
dropped = before - len(vg_raw)
print(f"Loaded {before:,} reviews, dropped {dropped:,} with missing text -> {len(vg_raw):,} retained")
print(vg_raw["Class"].value_counts().sort_index().to_string())

# -- 2. Split -------------------------------------------------------------------
splits_vg = stratified_split(
    vg_raw, label_col="Class", test_size=TEST_SIZE, val_size=VAL_SIZE, seed=SEED
)

# -- 3. Save --------------------------------------------------------------------
for name, df in splits_vg.items():
    out = SPLITS_DIR / f"vg_{name}.csv"
    df.to_csv(out, index=False)
    print(f"  Saved -> {out}")

Loaded 2,563,629 reviews
Class
1     311808
2     141306
3     212302
4     412276
5    1485937
Stratified split — total: 2,563,629  |  seed=1
  train:  2,076,539  (81.0 %)  [1=252,564  2=114,458  3=171,965  4=333,943  5=1,203,609]
  val  :    230,727  ( 9.0 %)  [1=28,063  2=12,718  3=19,107  4=37,105  5=133,734]
  test :    256,363  (10.0 %)  [1=31,181  2=14,130  3=21,230  4=41,228  5=148,594]
  Saved → ..\data\splits\vg_train.csv
  Saved → ..\data\splits\vg_val.csv
  Saved → ..\data\splits\vg_test.csv


In [8]:
splits_vg['train'].head()

,Sentence,Class
0,Great Game,5
1,Mine died and I bought this one and Im in busi...,5
2,I LOVE IT,5
3,Fast shipping and works great,5
4,Takes longer to charge but that's because it h...,5


## Sanity Checks

Verify that all nine files exist and have the expected row counts.

In [9]:
print('Split file summary\n' + '-' * 60)
for f in sorted(SPLITS_DIR.glob('*.csv')):
    df     = pd.read_csv(f)
    counts = df['Class'].value_counts().sort_index()
    dist   = '  '.join(f'{k}={v:,}' for k, v in counts.items())
    print(f'{f.name:<24}  rows={len(df):>10,}  [{dist}]')

print('\nAll files verified.')

Split file summary
------------------------------------------------------------
1k_test.csv               rows=       100  [0=50  1=50]
1k_train.csv              rows=       810  [0=405  1=405]
1k_val.csv                rows=        90  [0=45  1=45]
25k_test.csv              rows=     2,500  [0=988  1=1,512]
25k_train.csv             rows=    20,250  [0=8,006  1=12,244]
25k_val.csv               rows=     2,250  [0=890  1=1,360]
vg_test.csv               rows=   256,363  [1=31,181  2=14,130  3=21,230  4=41,228  5=148,594]
vg_train.csv              rows= 2,076,539  [1=252,564  2=114,458  3=171,965  4=333,943  5=1,203,609]
vg_val.csv                rows=   230,727  [1=28,063  2=12,718  3=19,107  4=37,105  5=133,734]

All files verified.


## How to Load the Splits in Other Notebooks

```python
import pandas as pd

# ── 1 K splits ────────────────────────────────────────────────────────────────────
train_1k = pd.read_csv('../data/splits/1k_train.csv')
val_1k   = pd.read_csv('../data/splits/1k_val.csv')
test_1k  = pd.read_csv('../data/splits/1k_test.csv')

# ── 25 K splits ────────────────────────────────────────────────────────────
train_25k = pd.read_csv('../data/splits/25k_train.csv')
val_25k   = pd.read_csv('../data/splits/25k_val.csv')
test_25k  = pd.read_csv('../data/splits/25k_test.csv')

# ── Video Games splits (Class: 1–5 star rating) ─────────────────────────
train_vg = pd.read_csv('../data/splits/vg_train.csv')
val_vg   = pd.read_csv('../data/splits/vg_val.csv')
test_vg  = pd.read_csv('../data/splits/vg_test.csv')
```

> Text is saved **raw**. Apply model-specific preprocessing after loading — e.g. call `preprocess_pandas` before TF-IDF, or pass raw text directly to a transformer tokenizer.